In [ ]:
ПРОБА RESNET18

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ============================================
# 1. УСТРОЙСТВО
# ============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

# ============================================
# 2. КЛАССЫ
# ============================================

class_names = ['Алевролит', 'Аргиллит', 'Глина', 'Переслаивание', 'Песчаник', 'Прочие', 'Углистые породы']

# ============================================
# 3. ТРАНСФОРМАЦИИ
# ============================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============================================
# 4. ЗАГРУЗКА МОДЕЛЕЙ
# ============================================

def load_model(light_type, model_path):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(class_names))
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model

model_ds = load_model("ДС", "models/resnet18_ДС_best.pth")
model_uv = load_model("УФ", "models/resnet18_УФ_best.pth")

print("✅ Модели загружены")

# ============================================
# 5. ПРОВЕРКА ФОТО ИЗ ПАПКИ manual
# ============================================

manual_dir = Path("Digital_core_v5_manual")

results = []

for light in ["ДС", "УФ"]:
    light_dir = manual_dir / light
    if not light_dir.exists():
        continue
    
    print(f"\n{'='*60}")
    print(f"ПРОВЕРКА {light} ФОТОГРАФИЙ")
    print(f"{'='*60}")
    
    # Выбираем модель
    model = model_ds if light == "ДС" else model_uv
    
    for class_dir in light_dir.iterdir():
        if not class_dir.is_dir():
            continue
        
        true_label = class_dir.name
        print(f"\n📁 Истинный класс: {true_label}")
        
        for img_path in class_dir.glob("*.jpg"):
            # Загружаем и предсказываем
            image = Image.open(img_path).convert('RGB')
            input_tensor = transform(image).unsqueeze(0).to(device)
            
            with torch.no_grad():
                output = model(input_tensor)
                probabilities = torch.nn.functional.softmax(output[0], dim=0)
                predicted_class = torch.argmax(probabilities).item()
                confidence = probabilities[predicted_class].item() * 100
                predicted_label = class_names[predicted_class]
            
            # Сохраняем результат
            results.append({
                'light': light,
                'true_label': true_label,
                'predicted_label': predicted_label,
                'confidence': confidence,
                'filename': img_path.name
            })
            
            # Выводим
            status = "✅" if true_label == predicted_label else "❌"
            print(f"\n  {status} {img_path.name}")
            print(f"    Предсказание: {predicted_label} ({confidence:.2f}%)")
            
            # Показываем топ-3
            top3 = torch.topk(probabilities, 3)
            top3_str = ', '.join([f'{class_names[idx]} ({prob*100:.1f}%)' for idx, prob in zip(top3.indices, top3.values)])
            print(f"    Топ-3: {top3_str}")
            
            # Показываем картинку
            plt.figure(figsize=(4, 4))
            plt.imshow(image)
            plt.title(f"True: {true_label}\nPred: {predicted_label} ({confidence:.1f}%)", 
                     color='green' if true_label == predicted_label else 'red')
            plt.axis('off')
            plt.tight_layout()
            plt.show()

# ============================================
# 6. ОБЩАЯ СТАТИСТИКА ПО РУЧНОЙ ПРОВЕРКЕ
# ============================================

print("\n" + "=" * 60)
print("ОБЩАЯ СТАТИСТИКА ПО РУЧНОЙ ПРОВЕРКЕ")
print("=" * 60)

correct = sum(1 for r in results if r['true_label'] == r['predicted_label'])
total = len(results)
print(f"Всего проверено: {total}")
print(f"Правильно: {correct} ({correct/total*100:.1f}%)")
print(f"Неправильно: {total - correct} ({(total-correct)/total*100:.1f}%)")

print("\nРЕЗУЛЬТАТЫ ПО ТИПАМ СВЕТА:")
for light in ["ДС", "УФ"]:
    light_results = [r for r in results if r['light'] == light]
    if light_results:
        light_correct = sum(1 for r in light_results if r['true_label'] == r['predicted_label'])
        print(f"  {light}: {light_correct}/{len(light_results)} ({light_correct/len(light_results)*100:.1f}%)")

print("\nРЕЗУЛЬТАТЫ ПО КЛАССАМ:")
for class_name in class_names:
    class_results = [r for r in results if r['true_label'] == class_name]
    if class_results:
        class_correct = sum(1 for r in class_results if r['true_label'] == r['predicted_label'])
        print(f"  {class_name}: {class_correct}/{len(class_results)} ({class_correct/len(class_results)*100:.1f}%)")

In [ ]:
ПРОБА Efficient

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ============================================
# 1. УСТРОЙСТВО
# ============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

# ============================================
# 2. КЛАССЫ
# ============================================

class_names = ['Алевролит', 'Аргиллит', 'Глина', 'Переслаивание', 'Песчаник', 'Прочие', 'Углистые породы']

# ============================================
# 3. ТРАНСФОРМАЦИИ (224x224 для совместимости с обученной моделью)
# ============================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============================================
# 4. ЗАГРУЗКА МОДЕЛЕЙ EFFICIENTNET-B3
# ============================================

def load_efficientnet(light_type, model_path):
    model = models.efficientnet_b3(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model

model_ds = load_efficientnet("ДС", "models/efficientnet_ДС_best.pth")
model_uv = load_efficientnet("УФ", "models/efficientnet_УФ_best.pth")

print("✅ Модели EfficientNet-B3 загружены")

# ============================================
# 5. ПРОВЕРКА ФОТО ИЗ ПАПКИ manual
# ============================================

manual_dir = Path("Digital_core_v5_manual")

results = []

for light in ["ДС", "УФ"]:
    light_dir = manual_dir / light
    if not light_dir.exists():
        continue
    
    print(f"\n{'='*60}")
    print(f"ПРОВЕРКА {light} ФОТОГРАФИЙ (EfficientNet-B3)")
    print(f"{'='*60}")
    
    # Выбираем модель
    model = model_ds if light == "ДС" else model_uv
    
    for class_dir in light_dir.iterdir():
        if not class_dir.is_dir():
            continue
        
        true_label = class_dir.name
        print(f"\n📁 Истинный класс: {true_label}")
        
        for img_path in class_dir.glob("*.jpg"):
            # Загружаем и предсказываем
            image = Image.open(img_path).convert('RGB')
            input_tensor = transform(image).unsqueeze(0).to(device)
            
            with torch.no_grad():
                output = model(input_tensor)
                probabilities = torch.nn.functional.softmax(output[0], dim=0)
                predicted_class = torch.argmax(probabilities).item()
                confidence = probabilities[predicted_class].item() * 100
                predicted_label = class_names[predicted_class]
            
            # Сохраняем результат
            results.append({
                'light': light,
                'true_label': true_label,
                'predicted_label': predicted_label,
                'confidence': confidence,
                'filename': img_path.name,
                'model': 'EfficientNet-B3'
            })
            
            # Выводим
            status = "✅" if true_label == predicted_label else "❌"
            print(f"\n  {status} {img_path.name}")
            print(f"    Предсказание: {predicted_label} ({confidence:.2f}%)")
            
            # Показываем топ-3
            top3 = torch.topk(probabilities, 3)
            top3_str = ', '.join([f'{class_names[idx]} ({prob*100:.1f}%)' for idx, prob in zip(top3.indices, top3.values)])
            print(f"    Топ-3: {top3_str}")
            
            # Показываем картинку
            plt.figure(figsize=(4, 4))
            plt.imshow(image)
            plt.title(f"EfficientNet: True={true_label}\nPred={predicted_label} ({confidence:.1f}%)", 
                     color='green' if true_label == predicted_label else 'red')
            plt.axis('off')
            plt.tight_layout()
            plt.show()

# ============================================
# 6. ОБЩАЯ СТАТИСТИКА ПО РУЧНОЙ ПРОВЕРКЕ
# ============================================

print("\n" + "=" * 60)
print("ОБЩАЯ СТАТИСТИКА (EfficientNet-B3)")
print("=" * 60)

correct = sum(1 for r in results if r['true_label'] == r['predicted_label'])
total = len(results)
print(f"Всего проверено: {total}")
print(f"Правильно: {correct} ({correct/total*100:.1f}%)")
print(f"Неправильно: {total - correct} ({(total-correct)/total*100:.1f}%)")

print("\nРЕЗУЛЬТАТЫ ПО ТИПАМ СВЕТА:")
for light in ["ДС", "УФ"]:
    light_results = [r for r in results if r['light'] == light]
    if light_results:
        light_correct = sum(1 for r in light_results if r['true_label'] == r['predicted_label'])
        print(f"  {light}: {light_correct}/{len(light_results)} ({light_correct/len(light_results)*100:.1f}%)")

print("\nРЕЗУЛЬТАТЫ ПО КЛАССАМ:")
for class_name in class_names:
    class_results = [r for r in results if r['true_label'] == class_name]
    if class_results:
        class_correct = sum(1 for r in class_results if r['true_label'] == r['predicted_label'])
        print(f"  {class_name}: {class_correct}/{len(class_results)} ({class_correct/len(class_results)*100:.1f}%)")

In [ ]:
ПРОБА MOBILE

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ============================================
# 1. УСТРОЙСТВО
# ============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")

# ============================================
# 2. КЛАССЫ
# ============================================

class_names = ['Алевролит', 'Аргиллит', 'Глина', 'Переслаивание', 'Песчаник', 'Прочие', 'Углистые породы']

# ============================================
# 3. ТРАНСФОРМАЦИИ
# ============================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============================================
# 4. ЗАГРУЗКА МОДЕЛЕЙ MOBILENETV3
# ============================================

def load_mobilenet(model_path):
    model = models.mobilenet_v3_large(weights=None)
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(class_names))
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model

model_ds = load_mobilenet("models/mobilenet_ДС_best.pth")
model_uv = load_mobilenet("models/mobilenet_УФ_best.pth")

print("✅ Модели MobileNetV3 загружены")

# ============================================
# 5. ПРОВЕРКА ФОТО ИЗ ПАПКИ manual
# ============================================

manual_dir = Path("Digital_core_v5_manual")

results = []

for light in ["ДС", "УФ"]:
    light_dir = manual_dir / light
    if not light_dir.exists():
        continue
    
    print(f"\n{'='*60}")
    print(f"ПРОВЕРКА {light} ФОТОГРАФИЙ (MobileNetV3)")
    print(f"{'='*60}")
    
    # Выбираем модель
    model = model_ds if light == "ДС" else model_uv
    
    for class_dir in light_dir.iterdir():
        if not class_dir.is_dir():
            continue
        
        true_label = class_dir.name
        print(f"\n📁 Истинный класс: {true_label}")
        
        for img_path in class_dir.glob("*.jpg"):
            # Загружаем и предсказываем
            image = Image.open(img_path).convert('RGB')
            input_tensor = transform(image).unsqueeze(0).to(device)
            
            with torch.no_grad():
                output = model(input_tensor)
                probabilities = torch.nn.functional.softmax(output[0], dim=0)
                predicted_class = torch.argmax(probabilities).item()
                confidence = probabilities[predicted_class].item() * 100
                predicted_label = class_names[predicted_class]
            
            # Сохраняем результат
            results.append({
                'light': light,
                'true_label': true_label,
                'predicted_label': predicted_label,
                'confidence': confidence,
                'filename': img_path.name,
                'model': 'MobileNetV3'
            })
            
            # Выводим
            status = "✅" if true_label == predicted_label else "❌"
            print(f"\n  {status} {img_path.name}")
            print(f"    Предсказание: {predicted_label} ({confidence:.2f}%)")
            
            # Показываем топ-3
            top3 = torch.topk(probabilities, 3)
            top3_str = ', '.join([f'{class_names[idx]} ({prob*100:.1f}%)' for idx, prob in zip(top3.indices, top3.values)])
            print(f"    Топ-3: {top3_str}")
            
            # Показываем картинку
            plt.figure(figsize=(4, 4))
            plt.imshow(image)
            plt.title(f"MobileNet: True={true_label}\nPred={predicted_label} ({confidence:.1f}%)", 
                     color='green' if true_label == predicted_label else 'red')
            plt.axis('off')
            plt.tight_layout()
            plt.show()

# ============================================
# 6. ОБЩАЯ СТАТИСТИКА ПО РУЧНОЙ ПРОВЕРКЕ
# ============================================

print("\n" + "=" * 60)
print("ОБЩАЯ СТАТИСТИКА (MobileNetV3)")
print("=" * 60)

correct = sum(1 for r in results if r['true_label'] == r['predicted_label'])
total = len(results)
print(f"Всего проверено: {total}")
print(f"Правильно: {correct} ({correct/total*100:.1f}%)")
print(f"Неправильно: {total - correct} ({(total-correct)/total*100:.1f}%)")

print("\nРЕЗУЛЬТАТЫ ПО ТИПАМ СВЕТА:")
for light in ["ДС", "УФ"]:
    light_results = [r for r in results if r['light'] == light]
    if light_results:
        light_correct = sum(1 for r in light_results if r['true_label'] == r['predicted_label'])
        print(f"  {light}: {light_correct}/{len(light_results)} ({light_correct/len(light_results)*100:.1f}%)")

print("\nРЕЗУЛЬТАТЫ ПО КЛАССАМ:")
for class_name in class_names:
    class_results = [r for r in results if r['true_label'] == class_name]
    if class_results:
        class_correct = sum(1 for r in class_results if r['true_label'] == r['predicted_label'])
        print(f"  {class_name}: {class_correct}/{len(class_results)} ({class_correct/len(class_results)*100:.1f}%)")